# Synthetic benchmark: Qwen3-1.7B base vs served v4

Colab: chọn GPU, mount Drive nếu cần, cấu hình bên dưới rồi chạy lần lượt. Cần bản sao repo cùng revision với service và một JSONL **chỉ gồm CV giả có nhãn gold**. Không dùng CV thật, token hay thông tin bí mật; tunnel công khai không bảo mật. Chỉ gửi dữ liệu synthetic qua tunnel và xác nhận server không log nội dung (`LOG_CV_CONTENT=false`).

JSONL theo `eval.dataset`: mỗi dòng có `input` (text) và `output` (profile JSON object hoặc JSON string); v4 builder cũng dùng format này. Dataset trong `ai-service/data/` bị gitignore, không mặc định giả định file có sẵn hoặc mọi split là synthetic. Tự kiểm tra nguồn và tách riêng synthetic validation có gold; không dùng `test_real`. Không có dữ liệu được nhúng trong notebook.

Base: `unsloth/Qwen3-1.7B`, transformers GPU, zero-shot, không adapter; cùng prompt serving v4 (`prompt_v3`), parser và evaluator repo. API đã chạy parser/postprocess riêng; notebook không thể tắt hoặc đảo ngược postprocess từ xa. Cấu hình server v4/postprocess phải được người chạy xác nhận. So sánh là **pipeline**, không phải ablation model thuần. API end-to-end gồm mạng, hàng đợi, server và retry; base là thời gian local generate gồm tokenization/decode, trên hardware/engine khác: không so tốc độ trực tiếp. Không in/lưu CV, gold hay output raw.


## Thiết lập repo public
Chạy ô sau để clone branch `hocnv/DO-06` vào `/content/recruit_system` nếu đường dẫn chưa tồn tại. Nếu đã tồn tại, notebook giữ nguyên, không pull hay ghi đè; hãy tự kiểm tra revision khớp service. Không đưa credential vào notebook hoặc URL.


In [ ]:
import os, subprocess, sys
REPO_ROOT = '/content/recruit_system'
if not os.path.lexists(REPO_ROOT):
    subprocess.run(['git', 'clone', '--branch', 'hocnv/DO-06', '--single-branch', 'https://github.com/long20102004/recruit_system.git', REPO_ROOT], check=True)
else:
    print('REPO_ROOT đã tồn tại; giữ nguyên, hãy kiểm tra revision.')


## Dữ liệu và cấu hình
Tự chuẩn bị JSONL validation **chỉ gồm CV synthetic có gold**, kiểm tra provenance từng nguồn trước khi xác nhận; không dùng CV thật hay `test_real`. Không giả định `val.jsonl` hoặc dữ liệu gitignored có sẵn, không commit dữ liệu vào repo. Đặt file ngoài repo: chọn upload ở ô tiếp theo hoặc mount Drive và điền đường dẫn. Không in nội dung CV, gold hoặc prediction.
Điền `API_URL` bằng HTTPS endpoint hiện tại kết thúc bằng `/api/v1/extract-cv`, không kèm credential hoặc query; không lưu URL tunnel tạm trong notebook chia sẻ. Chỉ gửi synthetic sau khi xác nhận server v4 không log nội dung.


In [ ]:
# Với Drive: bỏ comment hai dòng sau, mount rồi điền đường dẫn bên dưới.
# from google.colab import drive
# drive.mount('/content/drive')
SYNTHETIC_JSONL = ''  # đường dẫn file trên Drive; để trống nếu dùng upload bên dưới
UPLOAD_SYNTHETIC_JSONL = False  # đặt True để chọn một file bằng Colab files.upload
API_URL = ''  # HTTPS endpoint hiện tại /api/v1/extract-cv, không chứa secret
N_SAMPLES = 20
TIMEOUT_SECONDS = 300
RETRIES = 2  # retry lỗi mạng / 429 / 5xx
MAX_NEW_TOKENS = 2048
CONFIRM_SYNTHETIC_ONLY = False  # đặt True sau khi xác minh provenance toàn bộ file
CONFIRM_SERVER_V4_AND_NO_CONTENT_LOGS = False  # xác nhận model adapter v4 và LOG_CV_CONTENT=false
OUTPUT_DIR = '/content/benchmark_aggregates'


In [ ]:
if UPLOAD_SYNTHETIC_JSONL:
    if not CONFIRM_SYNTHETIC_ONLY:
        raise RuntimeError('Xác minh provenance synthetic toàn bộ file và xác nhận trước khi upload.')
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Hãy upload đúng một file JSONL synthetic có gold.')
    SYNTHETIC_JSONL = os.path.abspath(next(iter(uploaded)))
    del uploaded  # không hiển thị nội dung file


## Cài dependencies
Chạy ô cài đặt hiện có trước khi tải model và đánh giá.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51,<5', 'accelerate', 'rapidfuzz', 'json-repair', 'requests'], check=True)


In [ ]:
import os, json, time, statistics, csv
from pathlib import Path
from urllib.parse import urlsplit
import requests, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

service = Path(REPO_ROOT) / 'ai-service'
if not (service / 'eval' / 'metrics.py').is_file():
    raise RuntimeError('Thiếu repo: kiểm tra REPO_ROOT.')
sys.path.insert(0, str(service))
from eval.dataset import load_split_with_stats
from eval.metrics import score_sample, aggregate
from app.services.prompt_v3 import render_prompt
from app.utils.json_parse import extract_json
from app.utils.text import clean_cv_text

if not CONFIRM_SYNTHETIC_ONLY or not CONFIRM_SERVER_V4_AND_NO_CONTENT_LOGS:
    raise RuntimeError('Cần xác nhận synthetic và server v4 không log nội dung trong ô cấu hình.')
if not SYNTHETIC_JSONL or not Path(SYNTHETIC_JSONL).is_file():
    raise RuntimeError('Cần cung cấp JSONL synthetic validation có gold.')
if not isinstance(N_SAMPLES, int) or N_SAMPLES < 1 or RETRIES < 0 or TIMEOUT_SECONDS <= 0:
    raise RuntimeError('Cấu hình số mẫu / timeout / retry không hợp lệ.')
u = urlsplit(API_URL)
if u.scheme != 'https' or not u.hostname or u.username or u.password or u.query or u.fragment or u.path != '/api/v1/extract-cv':
    raise RuntimeError('API_URL phải là HTTPS endpoint /api/v1/extract-cv không kèm credential/query.')
if not torch.cuda.is_available():
    raise RuntimeError('Cần Colab GPU.')
try:
    samples, skipped = load_split_with_stats(SYNTHETIC_JSONL)
except Exception:
    raise RuntimeError('Không đọc được JSONL; kiểm tra format và nguồn synthetic.') from None
if skipped or len(samples) < N_SAMPLES or any(not s.cv_text.strip() or not isinstance(s.raw.get('output'), (str, dict)) for s in samples):
    raise RuntimeError('JSONL có dòng lỗi/rỗng hoặc không đủ mẫu; sửa dữ liệu trước khi chạy.')
samples = samples[:N_SAMPLES]
print('Số mẫu hợp lệ:', len(samples))


## Health check và model
Health xác nhận prompt version, không chứng minh checkpoint hay trạng thái postprocess. Người vận hành phải xác minh adapter v4 và cấu hình logging trước khi chạy. Server có thể ghi log riêng; notebook không thể kiểm soát từ xa.


In [ ]:
session = requests.Session()
session.trust_env = False  # không tự gửi proxy credential
try:
    health = session.get(f'{u.scheme}://{u.netloc}/health', timeout=TIMEOUT_SECONDS, allow_redirects=False)
    health.raise_for_status()
    h = health.json()
    if h.get('status') != 'ok' or h.get('prompt_version') != 'v4':
        raise ValueError('unhealthy')
except Exception:
    raise RuntimeError('Health check thất bại hoặc prompt_version không phải v4; kiểm tra endpoint.') from None
print('Health: OK; prompt v4. Engine:', h.get('engine') if h.get('engine') in ('llama_cpp', 'transformers') else 'unknown')
model_name = 'unsloth/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto')
model.eval()
print('Base loaded on GPU:', torch.cuda.get_device_name(0))


## Đánh giá
Validity là object có nội dung sau parser như serving; output lỗi hoặc request thất bại được tính invalid và chấm như profile rỗng. `eval.metrics` chuẩn hóa cả gold và prediction, micro-F1 cho các list field; scalar chỉ có accuracy, không tự chế F1. `experience.job_title` chỉ chấm các cặp company khớp. Các field khác không được evaluator hỗ trợ không được chấm. Base không chạy postprocess/continuation của server; API có thể có cả hai.


In [ ]:
def has_content(value):
    if isinstance(value, dict): return any(has_content(v) for v in value.values())
    if isinstance(value, list): return any(has_content(v) for v in value)
    if isinstance(value, str): return bool(value.strip())
    return value is not None and value is not False

def parse_profile(text):
    try:
        p = extract_json(text)
        return p if isinstance(p, dict) and has_content(p) else None
    except Exception:
        return None

def base_predict(cv):
    started = time.perf_counter()
    try:
        prompt = render_prompt(clean_cv_text(cv))
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, repetition_penalty=1.0)
        torch.cuda.synchronize()
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        if not text.startswith('{'): text = '{\n' + text
        return parse_profile(text), time.perf_counter() - started
    except Exception:
        return None, time.perf_counter() - started

def api_predict(cv):
    started = time.perf_counter()
    for attempt in range(RETRIES + 1):
        try:
            response = session.post(API_URL, json={'cv_text': cv}, timeout=TIMEOUT_SECONDS, allow_redirects=False)
            if response.status_code == 429 or response.status_code >= 500:
                if attempt < RETRIES:
                    time.sleep(min(2 ** attempt, 8)); continue
            if response.status_code != 200: break
            envelope = response.json()
            data = envelope.get('data') if isinstance(envelope, dict) else None
            if not isinstance(data, dict): break
            return parse_profile(json.dumps(data, ensure_ascii=False)), time.perf_counter() - started
        except (requests.RequestException, ValueError):
            if attempt < RETRIES:
                time.sleep(min(2 ** attempt, 8)); continue
            break
    return None, time.perf_counter() - started

scores = {'base': [], 'v4_api': []}
latencies = {'base': [], 'v4_api': []}
for i, sample in enumerate(samples, 1):
    for name, predict in (('base', base_predict), ('v4_api', api_predict)):
        prediction, elapsed = predict(sample.cv_text)
        scores[name].append(score_sample(prediction, sample.gold, sample.cv_text))
        latencies[name].append(elapsed)
        del prediction
    print(f'Đã xử lý {i}/{len(samples)}', flush=True)


## Export aggregate
Chỉ xuất số liệu tổng hợp: không có mẫu, ID, đường dẫn dataset, URL, CV hoặc prediction. Latency bao gồm cả lần lỗi và thời gian retry; không bao gồm tải model. Hãy tải `metrics.json` và `metrics.csv` về và gửi lại hai file để phân tích. Không suy diễn chất lượng hay hiệu năng trước khi chạy.


In [ ]:
results = {name: aggregate(scores[name], latencies[name]) for name in scores}
output = Path(OUTPUT_DIR)
output.mkdir(parents=True, exist_ok=True)
with (output / 'metrics.json').open('w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2, allow_nan=False)
with (output / 'metrics.csv').open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['pipeline', 'field', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'accuracy', 'n', 'valid_rate', 'latency_mean_s', 'latency_p50_s', 'latency_p95_s'])
    writer.writeheader()
    for name, result in results.items():
        latency = result['latency'] or {}
        common = {'pipeline': name, 'n': result['n'], 'valid_rate': result['valid_rate'], 'latency_mean_s': latency.get('mean'), 'latency_p50_s': latency.get('p50'), 'latency_p95_s': latency.get('p95')}
        for field, metrics in result['fields'].items():
            if field == 'languages': continue
            writer.writerow({**common, 'field': field, 'precision': metrics['p'], 'recall': metrics['r'], 'f1': metrics['f1'], 'tp': metrics['tp'], 'fp': metrics['fp'], 'fn': metrics['fn']})
        for field, metrics in result['scalars'].items():
            if not field.startswith('personal_info.'): continue
            writer.writerow({**common, 'field': field, 'accuracy': metrics['accuracy']})
print('Đã xuất metrics.json và metrics.csv; tải hai file và gửi lại.')
# Colab: from google.colab import files; files.download(str(output / 'metrics.json'))
# files.download(str(output / 'metrics.csv'))
